# 🕸️ 07: 구조화된 Context - GraphRAG 개념 데모

---

| 항목 | 내용 |
|------|------|
| **목표** | 관계가 중요한 문제에서 단순 top-k 검색만으로는 부족하다는 것을 이해한다 |
| **예상 실행 시간** | ⏱️ 빠른 시연 5분 / 전체 12분 |
| **API 키** | ❌ 불필요 (networkx + 로컬 임베딩) |
| **이전 노트북과의 연결** | Agentic Search까지 봤다. 관계가 복잡한 문제는 구조화된 context가 추가로 도움이 된다. |
| **고급 옵션** | 이 노트북은 선택적 심화 내용입니다. 시간이 부족하면 건너뛸 수 있습니다. |

---

## ⚠️ 중요: 올바른 기대치 설정

> **GraphRAG는 '정답'이 아닙니다.**  
> **관계가 중요한 문제에서 구조화된 context가 도움이 된다는 것이 핵심 메시지입니다.**

```
질문: "보안팀 이민준이 관여한 이슈들과 그로 인한 영향을 알려줘"

단순 검색 한계:
  - '이민준'이 포함된 문서 → 회의록만 찾음
  - 하지만 이민준이 담당한 SEC-POL-003이 실제 영향을 미친
    CloudSync v2.3 릴리즈와의 연결은 찾을 수 없음

구조화된 접근:
  사람 → 담당 문서 → 그 문서가 영향 준 제품/팀 순으로 추적
```

In [ ]:
!pip install -q networkx sentence-transformers openai
print("✅ 완료")

In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx
from IPython.display import display, HTML

# .env 에서 API 키 로드 (python-dotenv 필요, Colab 에서는 직접 입력 가능)
try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / ".env").exists():
            load_dotenv(_p / ".env"); break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

def setup(key=""):
    k = key or os.environ.get("OPENAI_API_KEY", "")
    if k and k not in ("", "sk-..."):
        try:
            from openai import OpenAI
            c = OpenAI(api_key=k)
            print("✅ API 모드")
            return "api", c
        except: pass
    print("✅ 로컬 모드 (networkx + sentence-transformers)")
    return "local", None

MODE, client = setup(OPENAI_API_KEY)

_st = None
def embed(texts):
    global _st
    if MODE == "api" and client:
        try:
            vecs = []
            for i in range(0, len(texts), 50):
                r = client.embeddings.create(input=texts[i:i+50], model="text-embedding-3-small")
                vecs.extend([x.embedding for x in r.data])
            return np.array(vecs, dtype=np.float32)
        except: pass
    if _st is None:
        print("📥 임베딩 모델 로딩...")
        from sentence_transformers import SentenceTransformer
        _st = SentenceTransformer("all-MiniLM-L6-v2")
        print("✅")
    return _st.encode(texts, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)

def call_llm(prompt, mock=None):
    if MODE == "api" and client:
        try:
            r = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role":"user","content":prompt}],
                temperature=0.3, max_tokens=600
            )
            return r.choices[0].message.content.strip()
        except Exception as e:
            print(f"⚠️ {e}")
    return f"[Mock]\n{mock}" if mock else "[로컬 모드]"

print(f"모드: {MODE}")

## 1️⃣ 엔티티-관계 데이터 구성

실제 GraphDB 없이 Python dict + networkx로 관계 그래프를 구성합니다.

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# ──────────────────────────────────────────────────
# 엔티티-관계 데이터 로드
# ──────────────────────────────────────────────────
from helpers.sample_data import ENTITIES, RELATIONS
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

# ──────────────────────────────────────────────────
# networkx 그래프 구성
# ──────────────────────────────────────────────────

G = nx.DiGraph()

for node_id, attrs in ENTITIES.items():
    G.add_node(node_id, **attrs)

for src, dst, rel in RELATIONS:
    G.add_edge(src, dst, relation=rel)

print(f"✅ 그래프 구성 완료")
print(f"   노드: {G.number_of_nodes()}개")
print(f"   엣지: {G.number_of_edges()}개")
print()
print("노드 유형별 수:")
from collections import Counter
type_counts = Counter(G.nodes[n]["type"] for n in G.nodes)
for t, cnt in type_counts.items():
    print(f"  {t}: {cnt}개")

## 2️⃣ 그래프 조회 함수

In [ ]:
def get_entity_context(entity_id: str, depth: int = 2) -> dict:
    """
    엔티티 주변의 관계를 depth만큼 탐색합니다.
    """
    if entity_id not in G:
        return {"error": f"{entity_id}를 찾을 수 없음"}

    entity_info = dict(G.nodes[entity_id])
    entity_info["id"] = entity_id

    # 아웃바운드 관계 (이 노드에서 나가는)
    outbound = []
    for _, dst, data in G.out_edges(entity_id, data=True):
        outbound.append({
            "target": dst,
            "relation": data.get("relation"),
            "target_type": G.nodes[dst].get("type"),
            "target_desc": G.nodes[dst].get("desc")
        })

    # 인바운드 관계 (이 노드로 들어오는)
    inbound = []
    for src, _, data in G.in_edges(entity_id, data=True):
        inbound.append({
            "source": src,
            "relation": data.get("relation"),
            "source_type": G.nodes[src].get("type"),
            "source_desc": G.nodes[src].get("desc")
        })

    # depth=2: 인접 노드의 관계도 가져오기
    if depth >= 2:
        extended = []
        for rel in outbound:
            target = rel["target"]
            for _, dst2, data2 in G.out_edges(target, data=True):
                if dst2 != entity_id:  # 자기 자신으로 돌아오는 관계 제외
                    extended.append({
                        "path": f"{entity_id} →[{rel['relation']}]→ {target} →[{data2['relation']}]→ {dst2}",
                        "end_type": G.nodes[dst2].get("type"),
                        "end_desc": G.nodes[dst2].get("desc")
                    })
        outbound_ext = extended
    else:
        outbound_ext = []

    return {
        "entity": entity_info,
        "direct_relations": outbound + inbound,
        "extended_paths": outbound_ext,
    }


def find_paths(start: str, end: str, max_depth: int = 3):
    """
    두 엔티티 간의 경로를 탐색합니다.
    """
    if start not in G or end not in G:
        return []
    try:
        paths = list(nx.all_simple_paths(G, start, end, cutoff=max_depth))
        return paths
    except nx.NetworkXNoPath:
        return []


def get_person_impact(person_id: str):
    """
    특정 사람이 담당한 문서들이 어떤 이슈/제품에 영향을 미쳤는지 추적합니다.
    """
    if person_id not in G:
        return {"error": f"{person_id}를 찾을 수 없음"}

    # 담당 문서들
    owned_docs = [
        (dst, data.get("relation"))
        for _, dst, data in G.out_edges(person_id, data=True)
        if G.nodes[dst].get("type") == "document"
    ]

    # 각 문서가 연결된 이슈와 제품
    impact_chain = []
    for doc_id, doc_rel in owned_docs:
        for _, downstream, data in G.out_edges(doc_id, data=True):
            downstream_type = G.nodes[downstream].get("type")
            impact_chain.append({
                "person": person_id,
                "doc": doc_id,
                "doc_relation": doc_rel,
                "downstream": downstream,
                "downstream_type": downstream_type,
                "downstream_rel": data.get("relation"),
                "downstream_desc": G.nodes[downstream].get("desc")
            })

    return {
        "person": person_id,
        "role": G.nodes[person_id].get("role"),
        "owned_docs": owned_docs,
        "impact_chain": impact_chain
    }


print("✅ 그래프 조회 함수 준비 완료")

## 3️⃣ 비교 데모: 단순 검색 vs 그래프 조회

In [ ]:
# ─────────────────────────────────────────────────────────
# 질문: "이민준이 관여한 변경사항이 어떤 제품에 영향을 미쳤나?"
# ─────────────────────────────────────────────────────────

QUERY = "이민준이 담당한 보안 정책이 어떤 제품과 이슈에 영향을 미쳤나?"

print("=" * 60)
print(f"  질문: '{QUERY}'")
print("=" * 60)

# 방법 1: 단순 벡터 검색
print("\n[방법 1] 단순 벡터 검색")
FLAT_DOCS = [
    {"id": "SEC-POL-003", "text": "API 키 관리 정책 v3.1. 담당: 이민준. 개발키 90일, 운영키 365일."},
    {"id": "MTG-SEC-REVIEW", "text": "보안 정책 개정 회의. 이민준 주도. v3.1 시행일 확정."},
    {"id": "MTG-2024Q3", "text": "전략 회의. 이민준 참석. CloudSync v2.3, DataPulse v1.1 출시 결정."},
    {"id": "REL-CLOUDSYNC-023", "text": "CloudSync v2.3. 보안 강화: OAuth 2.0 전환."},
    {"id": "REL-DATAPULSE-011", "text": "DataPulse v1.1. Kafka, Slack 알림. 보안 정책 적용."},
    {"id": "FAQ-SECURITY-001", "text": "보안 FAQ. 이민준 담당. API 키 유출 대응 절차."},
]

flat_vecs = embed([d["text"] for d in FLAT_DOCS])
q_vec = embed([QUERY])[0]
from numpy import dot
flat_scores = [float(dot(q_vec, v) / (np.linalg.norm(q_vec) * np.linalg.norm(v))) for v in flat_vecs]
sorted_flat = sorted(zip(flat_scores, FLAT_DOCS), reverse=True)

print("  상위 3개 검색 결과:")
for score, doc in sorted_flat[:3]:
    print(f"  [{doc['id']}] 점수: {score:.4f} - {doc['text'][:60]}")

print("\n  ⚠️ 단순 검색의 한계:")
print("     '이민준'이 포함된 문서를 찾지만,")
print("     이민준 → SEC-POL-003 → CloudSync/DataPulse 영향 체인은 보이지 않습니다.")

In [ ]:
# 방법 2: 그래프 조회
print("[방법 2] 그래프 조회 - 이민준의 영향 체인")
print()

impact = get_person_impact("이민준")

print(f"  👤 {impact['person']} ({impact['role']})")
print(f"\n  담당 문서:")
for doc_id, rel in impact["owned_docs"]:
    desc = ENTITIES[doc_id]["desc"]
    print(f"    → [{rel}] {doc_id}: {desc}")

print(f"\n  영향 체인 (이민준 → 문서 → 다운스트림):")

# 테이블로 정리
rows = []
for chain in impact["impact_chain"]:
    rows.append({
        "담당 문서": chain["doc"],
        "관계": chain["doc_relation"],
        "영향 대상": chain["downstream"],
        "대상 유형": chain["downstream_type"],
        "설명": chain["downstream_desc"][:40] if chain["downstream_desc"] else ""
    })

df = pd.DataFrame(rows).drop_duplicates()
display(df)

print("\n  ✅ 그래프 조회의 강점:")
print("     이민준 담당 SEC-POL-003이 CloudSync, DataPulse에 영향을 미쳤음을")
print("     명시적 관계 추적으로 발견할 수 있습니다.")

In [ ]:
# 경로 탐색: 이민준과 CloudSync 사이의 연결 고리
print("[경로 탐색] 이민준 → CloudSync")
print()

paths = find_paths("이민준", "CloudSync", max_depth=4)

if paths:
    print(f"  발견된 경로: {len(paths)}개")
    for i, path in enumerate(paths, 1):
        path_with_rels = []
        for j in range(len(path) - 1):
            src = path[j]
            dst = path[j+1]
            rel = G.edges[src, dst].get("relation", "?")
            path_with_rels.append(f"{src} -[{rel}]→")
        path_with_rels.append(path[-1])
        print(f"  경로 {i}: {' '.join(path_with_rels)}")
else:
    print("  직접 경로 없음 (비방향성 탐색이 필요할 수 있음)")

# undirected 그래프로 경로 찾기
UG = G.to_undirected()
try:
    shortest = nx.shortest_path(UG, "이민준", "CloudSync")
    print(f"\n  최단 경로 (방향 무시): {' → '.join(shortest)}")
    print(f"  경로 길이: {len(shortest)-1}홉")
except nx.NetworkXNoPath:
    print("  경로를 찾을 수 없습니다.")

## 4️⃣ 그래프 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 노드 색상 매핑
color_map = {
    "person": "#3498db",
    "document": "#27ae60",
    "product": "#e74c3c",
    "issue": "#f39c12"
}

node_colors = [color_map.get(G.nodes[n].get("type", ""), "gray") for n in G.nodes()]

plt.figure(figsize=(16, 10))
pos = nx.spring_layout(G, seed=42, k=2.5)

# 노드 그리기
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1500, alpha=0.9)
nx.draw_networkx_labels(G, pos, font_size=7, font_family="sans-serif")

# 엣지 그리기
edge_labels = {(s, t): d["relation"] for s, t, d in G.edges(data=True)}
nx.draw_networkx_edges(G, pos, edge_color="#888", arrows=True,
                        arrowsize=20, width=1.5, connectionstyle="arc3,rad=0.1")
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=5.5, alpha=0.7)

# 범례
patches = [
    mpatches.Patch(color=c, label=t)
    for t, c in color_map.items()
]
plt.legend(handles=patches, loc="upper left", fontsize=9)
plt.title("테크코어 내부 지식 그래프", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

print("💡 이 그래프를 보면 단순 문서 검색으로는 볼 수 없는")
print("   '이민준 → 보안정책 → CloudSync/DataPulse' 연결 고리가 보입니다.")

## 5️⃣ 그래프 기반 Context로 LLM 답변 생성

In [ ]:
def build_graph_context(query_entity: str) -> str:
    """그래프 조회 결과를 LLM 프롬프트용 텍스트로 변환합니다."""
    impact = get_person_impact(query_entity)
    ctx = [f"엔티티: {query_entity} ({impact.get('role', '')})"]
    ctx.append(f"담당 문서:")
    for doc_id, rel in impact.get("owned_docs", []):
        desc = ENTITIES.get(doc_id, {}).get("desc", "")
        ctx.append(f"  - {doc_id} ({rel}): {desc}")
    ctx.append("영향 체인:")
    for chain in impact.get("impact_chain", []):
        ctx.append(f"  - {chain['doc']} → {chain['downstream']} ({chain['downstream_rel']}): {chain['downstream_desc']}")
    return "\n".join(ctx)


graph_ctx = build_graph_context("이민준")

prompt = f"""아래 지식 그래프 정보를 바탕으로 질문에 답하세요.

{graph_ctx}

질문: 이민준이 담당한 보안 정책이 회사 제품들에 미친 영향을 정리해줘.

답변:"""

mock_answer = """이민준(보안팀)이 담당한 SEC-POL-003 v3.1 보안 정책은 다음과 같은 영향을 미쳤습니다:

1. CloudSync에 영향:
   - CloudSync v2.3에서 OAuth 2.0으로 인증 방식 변경 → 보안 정책과 일관된 인증 체계 수립
   - 모든 CloudSync API 키도 새 정책(개발 90일, 운영 365일)에 따라 갱신 필요

2. DataPulse에 영향:
   - DataPulse Kafka 커넥터의 API 인증 방식도 새 보안 정책 적용 대상
   - 기존 DataPulse 운영 키는 10월 일괄 감사 후 새 유효기간으로 재설정

3. 전사적 영향:
   - API-키-정책-강화 이슈가 모든 팀에 전파됨
   - MTG-SEC-REVIEW 회의를 통해 이 변경이 공식 결정됨

단순 문서 검색으로는 이민준 → SEC-POL-003 → CloudSync/DataPulse 연결 고리를
명시적으로 파악하기 어렵지만, 그래프 구조로는 명확히 추적됩니다."""

answer = call_llm(prompt, mock=mock_answer)
print("=" * 60)
print("  그래프 Context 기반 LLM 답변")
print("=" * 60)
print(answer)

## 6️⃣ 언제 구조화된 접근이 필요한가?

In [ ]:
display(HTML("""
<div style="font-family:Arial,sans-serif;max-width:820px;margin:10px auto;">
  <h3 style="color:#2c3e50;">검색 방식 선택 가이드</h3>
  <table style="width:100%;border-collapse:collapse;font-size:13px;">
    <thead>
      <tr style="background:#2c3e50;color:white;">
        <th style="padding:10px;">질문 유형</th>
        <th style="padding:10px;">권장 방식</th>
        <th style="padding:10px;">예시</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">단일 사실 조회</td>
        <td style="padding:10px;">벡터 검색 (RAG)</td>
        <td style="padding:10px;">"API 키 유효기간은?"</td>
      </tr>
      <tr>
        <td style="padding:10px;">비교/통합 질문</td>
        <td style="padding:10px;">Hybrid + Agentic</td>
        <td style="padding:10px;">"v2.3과 v2.2의 차이점은?"</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">관계 추적</td>
        <td style="padding:10px;">그래프 조회</td>
        <td style="padding:10px;">"이민준이 영향을 미친 이슈들은?"</td>
      </tr>
      <tr>
        <td style="padding:10px;">영향도 분석</td>
        <td style="padding:10px;">그래프 탐색</td>
        <td style="padding:10px;">"A 변경이 어떤 팀에 영향을 주나?"</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">계층 구조 질문</td>
        <td style="padding:10px;">그래프 + RAG 혼합</td>
        <td style="padding:10px;">"김도현 팀에서 최근에 한 것들은?"</td>
      </tr>
    </tbody>
  </table>
  <div style="margin-top:12px;padding:12px;background:#fff9c4;border-left:4px solid #f39c12;font-size:13px;">
    ⚠️ <strong>현실적 조언</strong>: GraphRAG는 강력하지만 구축 비용이 큽니다.<br>
    대부분의 경우 Hybrid RAG + Agentic Search로 충분히 해결됩니다.<br>
    관계 구조가 핵심인 도메인 (조직도, 지식 그래프, 규정 계층)에서만 도입을 고려하세요.
  </div>
</div>
"""))

---

## 🎤 강의자 멘트 포인트

> **"GraphRAG를 소개하는 이유는 '이게 표준 정답이다'가 아닙니다.**  
> **'문서 조각을 찾는 것'과 '관계를 추적하는 것'은 근본적으로 다른 문제라는 점입니다.**  
>
> '이민준이 담당한 정책이 어떤 제품에 영향을 미쳤나?'라는 질문은  
> 벡터 검색만으로는 완전히 답하기 어렵습니다.  
> 이민준이라는 사람과 정책, 정책과 제품 사이의 관계가 필요하니까요.  
>
> 여러분이 AI 시스템을 설계할 때,  
> '내 도메인에서 관계가 중요한가?'를 먼저 물어보세요.  
> 관계가 중요하다면, 단순 벡터 검색 이상의 접근이 필요할 수 있습니다."

## 🙋 청중 질문 유도
> - "여러분 회사에서 '관계 추적'이 중요한 도메인이 있나요? (조직도, 규정 계층, 의존성 관계)"
> - "GraphRAG를 쓰면 무엇이 좋고, 무엇이 어려울까요?"
> - "이 그래프는 누가 관리해야 할까요? 자동으로 업데이트될 수 있을까요?"

## 🏗️ 실무 확장 포인트
- **실제 GraphDB**: Neo4j, Amazon Neptune, TigerGraph
- **자동 엔티티 추출**: LLM으로 문서에서 엔티티-관계 자동 추출
- **Microsoft GraphRAG**: 오픈소스 구현체 (무겁지만 참고용)
- **Hybrid**: 벡터 검색 + 그래프 조회 결합 (가장 강력)

## ➕ 추가 실험
1. 새 엔티티와 관계를 추가해서 그래프 확장해보기
2. `김도현` 기준으로 impact chain 조회해보기
3. 두 제품(CloudSync, DataPulse) 사이의 공통 연결 노드 찾기